# DunkAPI: a season of basketball in five minutes

This notebook loads the free DunkAPI season pack (the last completed season, as CSV), answers a few questions offline, then asks the live API the same kind of question with a free key.

Data: DunkAPI (dunkapi.com), under the [DunkAPI Sample Data Licence](https://github.com/sports-data-hq/dunkapi-mcp/blob/main/DATA-LICENSE.md).

In [ ]:
import io, zipfile, urllib.request
import pandas as pd

URL = 'https://github.com/sports-data-hq/dunkapi-mcp/releases/download/season-pack/dunkapi-season-pack.zip'
z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read()))
tables = {n[:-4]: pd.read_csv(z.open(n)) for n in z.namelist() if n.endswith('.csv')}

def t(league, kind):
    """One table, e.g. t('nba', 'players')."""
    return next(df for name, df in tables.items() if f'-{league}-' in name and name.endswith('-' + kind))

sorted(tables)[:12]

## The final standings

In [ ]:
t('nba', 'standings').head(10)

## Shot-making: points over expected from shot location, minimum 500 shots

In [ ]:
x = t('nba', 'xpts')
x[x.shots >= 500].sort_values('points_over_expected', ascending=False)[['player', 'team', 'shots', 'expected_points_per_shot', 'points_per_shot', 'points_over_expected']].head(10)

## Teams by our end-of-season Elo

In [ ]:
t('nba', 'teams').sort_values('elo_end_of_season', ascending=False)[['team', 'games', 'ppg', 'elo_end_of_season']].head(10)

## The same, live

Every earlier season, play-by-play, lineups, odds and today's games are on the DunkAPI API. Get a free key and paste it below: one key for every sport at [app.sportsdatahq.com](https://app.sportsdatahq.com/signup?utm_source=colab&utm_medium=notebook&utm_campaign=dunkapi-notebook) (1,000 credits a month), which this notebook uses.

In [ ]:
import getpass, json, urllib.request
KEY = getpass.getpass('API key: ')
req = urllib.request.Request('https://api.sportsdatahq.com/v1/nba/get_standings', data=b'{}',
    headers={'x-api-key': KEY, 'content-type': 'application/json'})
res = urllib.request.urlopen(req)
print('credits used:', res.headers.get('x-credits-used'), '| left:', res.headers.get('x-credits-remaining'))
json.loads(res.read())['data']

Ask the same questions in plain English: add the MCP server `https://mcp.sportsdatahq.com/mcp` to Claude or ChatGPT, or install the basketball skills from https://github.com/sports-data-hq/dunkapi-mcp.